# Group-level cross-structure OFF period statistics

Aggregate cross-structure OFF overlap statistics across all qualifying multi-cortical
subjects and produce group-level plots. Each observation is a (subject, structure,
condition) tuple.

Source data: per-subject outputs from `cross_structure_offs.do_subject()`.

Prerequisite: `cross_structure_offs.do_experiment()` must have been run.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pubplots as pp
import seaborn as sns
from scipy.stats import spearmanr

from cnpix_local_sleep import const, plots
from cnpix_local_sleep.morphological.pipeline import cross_structure_offs

warnings.filterwarnings("ignore", category=PendingDeprecationWarning)

In [ ]:
# --- OFF source & single-condition scope selectors ---
# OFF_SOURCE drives both the pipeline outputs read below and the per-OFF
# properties. One of: "morphological-full48h" (default), "morphological".
OFF_SOURCE = "morphological-full48h"

# SINGLE_COND_SCOPE controls the single-slice group plots (1a, 1b, 4a):
#   "whole_recording" (default) -> pool all OFFs across the 48h recording
#                                  (requires OFF_SOURCE="morphological-full48h")
#   "condition"                 -> each plot's named statistical condition
SINGLE_COND_SCOPE = "whole_recording"

assert OFF_SOURCE in cross_structure_offs.OFF_SOURCES, OFF_SOURCE
assert SINGLE_COND_SCOPE in ("whole_recording", "condition")
assert SINGLE_COND_SCOPE != "whole_recording" or OFF_SOURCE == "morphological-full48h", (
    "SINGLE_COND_SCOPE='whole_recording' requires OFF_SOURCE='morphological-full48h'"
)

conditions = list(const.CORE_CONDITIONS)
palette = plots.get_condition_palette()
condition_order = list(const.CORE_CONDITIONS)

NOTEBOOK_NAME = "group_cross_structure_offs"
OUTPUT_DIR = Path(f"./outputs/{NOTEBOOK_NAME}/{OFF_SOURCE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OFF_SOURCE={OFF_SOURCE}  SINGLE_COND_SCOPE={SINGLE_COND_SCOPE}")
print(f"Saving SVGs to {OUTPUT_DIR}")

In [ ]:
# Ensure the per-subject pipeline outputs exist for this OFF_SOURCE. Idempotent:
# do_subject SKIPs subjects whose outputs already exist. no_jitter=True keeps it
# cheap (the group plots below don't use the jitter null, which is also invalid
# for absolute-time full-48h OFFs).
cross_structure_offs.do_experiment(off_source=OFF_SOURCE, no_jitter=True)

In [ ]:
subjects = cross_structure_offs.get_multi_cortical_subjects(OFF_SOURCE)
print(f"{len(subjects)} qualifying subjects: {subjects}")

## Load and concatenate per-subject outputs

For each qualifying subject, load:
- `condition_comparison.parquet` -- (condition, structure) summary stats
- `overlap_counts.parquet` -- per-OFF overlap annotations
- `chance_baselines.parquet` -- observed vs expected local fractions

Add a `subject` column to each and concatenate across subjects.

In [ ]:
comparison_dfs = []
for subject in subjects:
    path = cross_structure_offs._get_output_path(
        subject, "condition_comparison.parquet", OFF_SOURCE
    )
    if not path.exists():
        print(f"WARNING: missing condition_comparison.parquet for {subject}")
        continue
    df = pd.read_parquet(path)
    df["subject"] = subject
    comparison_dfs.append(df)

group_comparison = pd.concat(comparison_dfs, ignore_index=True)
print(f"group_comparison: {len(group_comparison)} rows")
print(f"Columns: {list(group_comparison.columns)}")
print(
    f"Unique (subject, structure) pairs: "
    f"{group_comparison.groupby(['subject', 'structure']).ngroups}"
)
group_comparison.head()

In [ ]:
overlap_dfs = []
for subject in subjects:
    path = cross_structure_offs._get_output_path(
        subject, "overlap_counts.parquet", OFF_SOURCE
    )
    if not path.exists():
        print(f"WARNING: missing overlap_counts.parquet for {subject}")
        continue
    overlap_dfs.append(pd.read_parquet(path))

# overlap_counts already carries `subject`, `start_time`, `end_time`.
group_overlaps = pd.concat(overlap_dfs, ignore_index=True)
print(f"group_overlaps: {len(group_overlaps)} rows")
print(f"Columns: {list(group_overlaps.columns)}")
group_overlaps.head()

In [ ]:
baseline_dfs = []
for subject in subjects:
    path = cross_structure_offs._get_output_path(
        subject, "chance_baselines.parquet", OFF_SOURCE
    )
    if not path.exists():
        print(f"WARNING: missing chance_baselines.parquet for {subject}")
        continue
    df = pd.read_parquet(path)
    df["subject"] = subject
    baseline_dfs.append(df)

group_baselines = pd.concat(baseline_dfs, ignore_index=True)
print(f"group_baselines: {len(group_baselines)} rows")
group_baselines.head()

In [ ]:
# Load per-OFF properties across structures (same OFF_SOURCE as the pipeline
# outputs) for plots 1 and 4. Merged with the overlap annotations on stable
# interval keys (start_time/end_time) rather than a positional index.
props = cross_structure_offs.load_cross_structure_offs(
    OFF_SOURCE, conditions=conditions
)
print(f"Loaded {len(props)} OFFs for {props['subject'].nunique()} subjects")

In [ ]:
# Merge overlap annotations onto per-OFF properties on stable keys.
_keys = ["subject", "structure", "condition", "start_time", "end_time"]
_left = props.astype({"subject": str, "structure": str, "condition": str})
_right = group_overlaps.astype({"subject": str, "structure": str, "condition": str})
annotated_offs = _left.merge(
    _right[_keys + ["n_overlapping_structures", "is_local"]],
    on=_keys,
    how="inner",
)
annotated_offs["overlap_status"] = annotated_offs["is_local"].map(
    {True: "Local", False: "Overlapping"}
)
print(
    f"Annotated OFFs: {len(annotated_offs)} "
    f"(matched {len(annotated_offs)}/{len(group_overlaps)} overlap records)"
)

In [ ]:
# Summary of subjects and structures.
summary = group_comparison.groupby("subject").agg(
    structures=("structure", lambda s: ", ".join(sorted(s.unique()))),
    n_structures=("structure", "nunique"),
    total_offs=("n_offs", "sum"),
)
print(summary.to_string())

In [ ]:
# --- Whole-recording vs per-condition scope for the single-slice plots ---
# Plots 1a/1b/4a show one slice. "condition" uses each plot's named condition;
# "whole_recording" pools all OFFs across the 48h recording. The pipeline only
# computes the six statistical conditions, so the whole-recording overlap
# annotations are computed here in-notebook.
property_cols = ["median_duration", "span", "area"]
available_cols = [c for c in property_cols if c in annotated_offs.columns]

if SINGLE_COND_SCOPE == "whole_recording":
    _parts = []
    for subject in subjects:
        s_offs = cross_structure_offs.load_whole_recording_offs(
            OFF_SOURCE, subject=subject
        )
        s_structs = sorted(s_offs["structure"].dropna().unique())
        if len(s_structs) < 2:
            continue
        s_cond = s_offs["condition"].iloc[0]  # "Full48h"
        s_dict = cross_structure_offs._build_offs_dict(s_offs, s_structs, [s_cond])
        s_counts = cross_structure_offs.compute_overlap_counts(
            s_dict, s_structs, s_cond
        )
        for struct in s_structs:
            g = s_dict[(struct, s_cond)].copy()
            if g.empty:
                continue
            g["n_overlapping_structures"] = s_counts[struct]
            g["is_local"] = s_counts[struct] == 0
            _parts.append(g)
    whole_annotated_offs = pd.concat(_parts, ignore_index=True)
    whole_annotated_offs["overlap_status"] = whole_annotated_offs["is_local"].map(
        {True: "Local", False: "Overlapping"}
    )
    print(
        f"whole_annotated_offs: {len(whole_annotated_offs)} OFFs across "
        f"{whole_annotated_offs['subject'].nunique()} subjects"
    )
else:
    whole_annotated_offs = None


def get_scope_subset(default_cond):
    """Return (subset_df, scope_cond, scope_label) for a single-slice plot."""
    if SINGLE_COND_SCOPE == "whole_recording":
        return whole_annotated_offs, "Full48h", "whole_recording"
    return (
        annotated_offs[annotated_offs["condition"] == default_cond],
        default_cond,
        default_cond,
    )

## Plot 1: OFF properties vs overlap degree

Subject-level: `plot_properties_vs_overlap_degree`, box plots of span_rel2max and
duration against overlap degree.

Group-level: (a) distribution of Spearman rho values across (subject, structure)
observations; (b) pooled box plot of property by overlap degree category.


In [ ]:
# 1a: Spearman rho distributions.
subset, cond, scope_label = get_scope_subset("Early.REC.NREM")
properties = ["span_rel2max", "duration"]

rho_records = []
for (subj, struct), grp in subset.groupby(["subject", "structure"], observed=True):
    if len(grp) < 10:
        continue
    for prop in properties:
        vals = grp.dropna(subset=[prop])
        if len(vals) < 10:
            continue
        rho, p = spearmanr(vals["n_overlapping_structures"], vals[prop])
        rho_records.append(
            {
                "subject": subj,
                "structure": struct,
                "property": prop,
                "rho": rho,
                "p_value": p,
                "n": len(vals),
            }
        )

rho_df = pd.DataFrame(rho_records)
print(f"{len(rho_df)} (subject, structure, property) observations")

with pp.destination("default"):
    fig, axes = plt.subplots(1, len(properties), figsize=(2.5 * len(properties), 4))
    for i, prop in enumerate(properties):
        ax = axes[i]
        prop_df = rho_df[rho_df["property"] == prop]
        sns.boxplot(
            data=prop_df, y="rho", ax=ax, fliersize=0, color=palette.get(cond, "gray")
        )
        sns.stripplot(data=prop_df, y="rho", ax=ax, color="k", alpha=0.7)
        ax.axhline(0, color="gray", ls="--", lw=0.8)
        ax.set_title(f"Spearman rho: overlap degree vs {prop}")
        ax.set_ylabel("Spearman rho")
        ax.set_xlabel("")
        median_rho = prop_df["rho"].median()
        ax.text(
            0.95,
            0.05,
            f"median={median_rho:.3f}\nn={len(prop_df)}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=8,
        )
    fig.suptitle(
        f"Overlap degree vs OFF properties ({cond})\nEach dot = one (subject, structure)",
        fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"cross_structure_1a_spearman_rho_{scope_label}.svg")
    plt.show()

In [ ]:
# 1b: Pooled box plot of property by overlap degree (uses subset/cond from 1a).
with pp.destination("default"):
    fig, axes = plt.subplots(1, 3, figsize=(9, 5))

    plot_offs = subset.copy()
    plot_offs["overlap_degree"] = (
        plot_offs["n_overlapping_structures"].clip(upper=3).astype(str)
    )
    plot_offs.loc[plot_offs["n_overlapping_structures"] >= 3, "overlap_degree"] = "3+"

    degree_order = [str(d) for d in range(3)] + ["3+"]
    n_degrees = len(degree_order)
    _flare_cmap = sns.color_palette("flare", as_cmap=True)
    _flare_vals = np.linspace(0.1, 0.9, n_degrees)
    degree_palette = {
        d: _flare_cmap(_flare_vals[i]) for i, d in enumerate(degree_order)
    }

    if "span_rel2max" in plot_offs.columns:
        ax = axes[0]
        sns.boxplot(
            data=plot_offs.dropna(subset=["span_rel2max"]),
            x="overlap_degree",
            y="span_rel2max",
            order=degree_order,
            ax=ax,
            fliersize=0,
            palette=degree_palette,
        )
        ax.set_xlabel("Overlap degree (# other structures)")
        ax.set_ylabel("span_rel2max")
        ax.set_title("Spatial span vs overlap degree")
    else:
        axes[0].set_visible(False)

    ax = axes[1]
    sns.boxplot(
        data=plot_offs,
        x="overlap_degree",
        y="duration",
        order=degree_order,
        ax=ax,
        fliersize=0,
        palette=degree_palette,
    )
    ax.set_yscale("log")
    ax.set_xlabel("Overlap degree (# other structures)")
    ax.set_ylabel("Duration (s)")
    ax.set_title("Duration vs overlap degree")

    if "area_rel2span" in plot_offs.columns:
        ax = axes[2]
        sns.boxplot(
            data=plot_offs.dropna(subset=["area_rel2span"]),
            x="overlap_degree",
            y="area_rel2span",
            order=degree_order,
            ax=ax,
            fliersize=0,
            palette=degree_palette,
        )
        ax.set_xlabel("Overlap degree (# other structures)")
        ax.set_ylabel("area_rel2span")
        ax.set_title("Area/span vs overlap degree")
        ax.set_ylim(0, 6)
    else:
        axes[2].set_visible(False)

    fig.suptitle(
        f"OFF properties vs overlap degree | {cond} | All subjects pooled", fontsize=12
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"cross_structure_1b_properties_vs_degree_{scope_label}.svg"
    )
    plt.show()

## Plot 2: local vs global OFFs

Subject-level: `plot_overlap_degree_bars`, a stacked bar chart of overlap degree
fractions by structure and condition.

Group-level: box and strip plots showing the distribution of `frac_local` and
`frac_any_overlap` across (subject, structure) observations, grouped by condition.


In [ ]:
with pp.destination("default"):
    fig, axes = plt.subplots(1, 2, figsize=(7, 5))

    for ax, var, title in zip(
        axes,
        ["frac_local", "frac_any_overlap"],
        ["Fraction local (no overlap)", "Fraction overlapping (>=1)"],
    ):
        plot_df = group_comparison[group_comparison["condition"].isin(condition_order)]
        sns.boxplot(
            data=plot_df,
            x="condition",
            y=var,
            order=condition_order,
            fliersize=0,
            hue="condition",
            palette=palette,
            ax=ax,
        )
        sns.stripplot(
            data=plot_df,
            x="condition",
            y=var,
            order=condition_order,
            color="k",
            alpha=0.6,
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel(None)
        ax.set_ylabel(var)
        ax.tick_params(axis="x", rotation=45)

    fig.suptitle(
        "Local vs Global OFFs across conditions\nEach dot = one (subject, structure)",
        fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cross_structure_2_local_vs_global.svg")
    plt.show()

## Plot 3: condition comparison

Subject-level: `plot_condition_comparison`, grouped bar charts of `frac_any_overlap` and
`mean_overlap_degree` by structure and condition.

Group-level: box and strip plots with all 6 conditions on the x-axis, showing individual
(subject, structure) observations, in two panels: fraction overlapping, and mean number
of overlapping structures.


In [ ]:
with pp.destination("default"):
    fig, axes = plt.subplots(1, 2, figsize=(7, 5))

    metrics = [
        ("frac_any_overlap", "Fraction overlapping"),
        ("mean_overlap_degree", "Mean # overlapping structures"),
    ]

    for ax, (var, ylabel) in zip(axes, metrics):
        plot_df = group_comparison[group_comparison["condition"].isin(condition_order)]
        sns.boxplot(
            data=plot_df,
            x="condition",
            y=var,
            order=condition_order,
            fliersize=0,
            hue="condition",
            palette=palette,
            ax=ax,
        )
        sns.stripplot(
            data=plot_df,
            x="condition",
            y=var,
            order=condition_order,
            color="k",
            alpha=0.6,
            ax=ax,
        )
        ax.set_ylabel(ylabel)
        ax.set_xlabel(None)
        ax.tick_params(axis="x", rotation=45)

    fig.suptitle(
        "Condition comparison: cross-structure overlap\nEach dot = one (subject, structure)",
        fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cross_structure_3_condition_comparison.svg")
    plt.show()

## Plot 4: OFF properties, local vs overlapping

Subject-level: `plot_local_vs_overlapping_properties`, split violin plots comparing
local and overlapping OFF properties per structure.

Group-level: compute the median property per (subject, structure, condition,
overlap_status), then show as box and strip plots.


In [ ]:
# Plot 4a: median property per (subject, structure, overlap_status) for one slice.
scope_offs_4a, cond, scope_label = get_scope_subset(conditions[0])
cols_4a = [c for c in property_cols if c in scope_offs_4a.columns]
agg_4a = (
    scope_offs_4a.groupby(
        ["subject", "structure", "condition", "overlap_status"], observed=True
    )[cols_4a]
    .median()
    .reset_index()
)

_flare_cmap = sns.color_palette("flare", as_cmap=True)
_flare_vals = np.linspace(0.1, 0.9, 2)
local_overlap_palette = {
    "Local": _flare_cmap(_flare_vals[0]),
    "Overlapping": _flare_cmap(_flare_vals[-1]),
}

plot_agg = agg_4a[agg_4a["condition"] == cond]

with pp.destination("default"):
    fig, axes = plt.subplots(
        1, len(cols_4a), figsize=(2.5 * len(cols_4a), 4), squeeze=False
    )
    for col_idx, prop in enumerate(cols_4a):
        ax = axes[0, col_idx]
        sns.boxplot(
            data=plot_agg,
            x="overlap_status",
            y=prop,
            order=["Local", "Overlapping"],
            fliersize=0,
            palette=local_overlap_palette,
            ax=ax,
        )
        sns.stripplot(
            data=plot_agg,
            x="overlap_status",
            y=prop,
            order=["Local", "Overlapping"],
            color="k",
            alpha=0.6,
            ax=ax,
        )
        if prop in ("area", "median_duration"):
            ax.set_yscale("log")
        ax.set_title(prop)
        ax.set_xlabel(None)

    fig.suptitle(
        f"OFF properties: Local vs Overlapping | {cond}\n"
        "Each dot = median for one (subject, structure)",
        fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"cross_structure_4a_local_vs_overlapping_{scope_label}.svg"
    )
    plt.show()

In [ ]:
# Plot 4b: all conditions, hue = overlap_status (per-condition annotations).
agg_all = (
    annotated_offs.groupby(
        ["subject", "structure", "condition", "overlap_status"], observed=True
    )[available_cols]
    .median()
    .reset_index()
)

with pp.destination("default"):
    fig, axes = plt.subplots(
        len(available_cols), 1, figsize=(4, 4 * len(available_cols)), squeeze=False
    )

    for row_idx, prop in enumerate(available_cols):
        ax = axes[row_idx, 0]
        plot_all = agg_all[agg_all["condition"].isin(condition_order)]
        sns.boxplot(
            data=plot_all,
            x="condition",
            y=prop,
            hue="overlap_status",
            hue_order=["Local", "Overlapping"],
            order=condition_order,
            fliersize=0,
            palette=local_overlap_palette,
            ax=ax,
        )
        sns.stripplot(
            data=plot_all,
            x="condition",
            y=prop,
            hue="overlap_status",
            hue_order=["Local", "Overlapping"],
            order=condition_order,
            palette="dark:black",
            dodge=True,
            alpha=0.5,
            legend=False,
            ax=ax,
        )
        if prop in ("area", "median_duration"):
            ax.set_yscale("log")
        ax.set_title(prop)
        ax.set_xlabel(None)
        ax.tick_params(axis="x", rotation=45)

    fig.suptitle(
        "OFF properties: Local vs Overlapping | All conditions\n"
        "Each dot = median for one (subject, structure)",
        fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "cross_structure_4b_local_vs_overlapping_all_conditions.svg"
    )
    plt.show()

## Supplementary: observed vs expected local fraction

From `chance_baselines.parquet`: compare the observed local fraction to what is expected
under an independence assumption. A ratio < 1 indicates more overlap than chance.


In [ ]:
with pp.destination("default"):
    fig, axes = plt.subplots(1, 2, figsize=(6, 5))

    plot_bl = group_baselines[group_baselines["condition"].isin(condition_order)]

    ax = axes[0]
    sns.boxplot(
        data=plot_bl,
        x="condition",
        y="ratio",
        order=condition_order,
        fliersize=0,
        hue="condition",
        palette=palette,
        ax=ax,
    )
    sns.stripplot(
        data=plot_bl,
        x="condition",
        y="ratio",
        order=condition_order,
        color="k",
        alpha=0.6,
        ax=ax,
    )
    ax.axhline(1.0, color="gray", ls="--", lw=0.8, label="Independence")
    ax.set_ylabel("Observed / Expected local fraction")
    ax.set_xlabel(None)
    ax.set_title("Chance baseline ratio")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=8)

    ax = axes[1]
    ax.scatter(
        plot_bl["expected_local_frac"],
        plot_bl["observed_local_frac"],
        c=[palette.get(c, "gray") for c in plot_bl["condition"]],
        alpha=0.6,
        edgecolors="k",
        linewidth=0.3,
    )
    lims = [0, 1]
    ax.plot(lims, lims, "k--", lw=0.8, label="y=x")
    ax.set_xlabel("Expected local fraction (independence)")
    ax.set_ylabel("Observed local fraction")
    ax.set_title("Observed vs expected local fraction")
    ax.legend(fontsize=8)

    fig.suptitle(
        "Chance baselines across subjects\nEach dot = one (subject, structure, condition)",
        fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cross_structure_supp_chance_baselines.svg")
    plt.show()

## Plot 5: excess globality, windowed local-shift null

Plots 1-4 show that more global (higher overlap-degree) OFFs tend to be larger. But
longer and denser OFFs collide with other structures more often by chance, so part of
that relationship is mechanical. Here we subtract the expected collisions with a
windowed local-shift null (`cross_structure_offs.compute_windowed_excess_globality`):

- Each focal OFF is held fixed while every partner structure's OFFs are locally jittered
  (±`window`/2, reflected at bout edges) within their own NREM bout, preserving each
  OFF's duration and the within-bout OFF density while destroying fine-timescale
  cross-structure alignment.
- `excess_globality = observed_degree - null_mean` is the overlap degree beyond what
  collision geometry buys, duration-corrected by construction.
- `EXCESS_NULL_SCOPE` selects the NREM domain: `"whole_recording"` (all 48 h NREM) or a
  single condition's NREM, e.g. `"Early.REC.NREM"`.

If the size/globality relationship were purely mechanical, `excess_globality` would be
about flat against span, duration and area. A positive trend is genuine coordination.
The analytic `E[degree | duration]` curve (independence, extended-interval) is overlaid
in 5b as a cross-check on the simulation.


In [ ]:
# Excess-globality config + compute. Requires OFF_SOURCE='morphological-full48h'.
EXCESS_NULL_SCOPE = "whole_recording"  # or a condition, e.g. "Early.REC.NREM"
EXCESS_WINDOW = 60.0  # local-shift window (s); 30-180 reasonable
EXCESS_N_SHUFFLES = 200

assert OFF_SOURCE == "morphological-full48h", (
    "Excess globality requires OFF_SOURCE='morphological-full48h'"
)

# Load whole-recording OFFs once for all subjects (avoids re-collecting per
# subject inside do_subject_excess_globality), then score each subject.
_all_whole = cross_structure_offs.load_whole_recording_offs(OFF_SOURCE)

excess_parts = []
for subject in subjects:
    s_offs = _all_whole[_all_whole["subject"] == subject]
    s_structs = sorted(s_offs["structure"].dropna().unique())
    if len(s_structs) < 2:
        continue
    ref_probe = s_offs["probe"].iloc[0]
    bouts = cross_structure_offs.get_nrem_bouts(subject, ref_probe, EXCESS_NULL_SCOPE)
    res = cross_structure_offs.compute_windowed_excess_globality(
        s_offs,
        s_structs,
        bouts=bouts,
        window=EXCESS_WINDOW,
        n_shuffles=EXCESS_N_SHUFFLES,
    )
    excess_parts.append(res)

excess_df = pd.concat(excess_parts, ignore_index=True)
print(
    f"excess_df: {len(excess_df)} OFFs across {excess_df['subject'].nunique()} "
    f"subjects | scope={EXCESS_NULL_SCOPE} window={EXCESS_WINDOW}s "
    f"shuffles={EXCESS_N_SHUFFLES}"
)
print(
    f"  mean observed={excess_df['observed_degree'].mean():.3f}  "
    f"null={excess_df['null_mean'].mean():.3f}  "
    f"analytic={excess_df['analytic_expected_degree'].mean():.3f}  "
    f"excess={excess_df['excess_globality'].mean():.3f}"
)
excess_df.head()

In [ ]:
# 5a: Does OFF size predict globality beyond chance collisions?
# Per (subject, structure): Spearman rho of each property vs the raw overlap
# degree (mechanical + genuine) and vs excess globality (collision-corrected).
# A raw rho that collapses toward 0 for excess means the size-globality link is
# largely mechanical; a surviving positive excess rho is genuine coordination.
excess_props = [c for c in ["span", "duration", "area"] if c in excess_df.columns]
targets = [("observed_degree", "raw degree"), ("excess_globality", "excess")]

rho5_records = []
for (subj, struct), grp in excess_df.groupby(["subject", "structure"], observed=True):
    if len(grp) < 10:
        continue
    for prop in excess_props:
        g = grp.dropna(subset=[prop])
        if len(g) < 10:
            continue
        for target, tlabel in targets:
            rho, _ = spearmanr(g[prop], g[target])
            rho5_records.append(
                {
                    "subject": subj,
                    "structure": struct,
                    "property": prop,
                    "target": tlabel,
                    "rho": rho,
                }
            )
rho5_df = pd.DataFrame(rho5_records)

_target_order = ["raw degree", "excess"]
with pp.destination("default"):
    fig, axes = plt.subplots(
        1, len(excess_props), figsize=(2 * len(excess_props), 4), squeeze=False
    )
    for i, prop in enumerate(excess_props):
        ax = axes[0, i]
        sub = rho5_df[rho5_df["property"] == prop]
        sns.boxplot(
            data=sub,
            x="target",
            y="rho",
            order=_target_order,
            color="k",
            fill=False,
            legend=False,
            ax=ax,
            fliersize=0,
        )
        sns.stripplot(
            data=sub,
            x="target",
            y="rho",
            order=_target_order,
            ax=ax,
            color="k",
            alpha=0.7,
        )
        ax.axhline(0, color="gray", ls="--", lw=0.8)
        ax.set_title(prop)
        ax.set_xlabel("")
        ax.set_ylabel("Spearman rho")
    fig.suptitle(
        "Size vs globality: raw degree vs collision-corrected excess\n"
        f"Each dot = one (subject, structure) | scope={EXCESS_NULL_SCOPE} "
        f"window={EXCESS_WINDOW:.0f}s",
        fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"cross_structure_5a_excess_rho_{EXCESS_NULL_SCOPE}.svg")
    plt.show()

In [ ]:
# 5b: Overlap degree vs OFF duration -- observed vs windowed null vs analytic.
# Duration-binned means pooled across subjects/structures. Where observed sits
# above the null / analytic collision expectation is the genuine excess; where
# they coincide, the size-globality link is mechanical.

import matplotlib

N_BINS = 15
dur = excess_df["duration"].to_numpy()
edges = np.unique(np.quantile(dur, np.linspace(0, 1, N_BINS + 1)))
centers = 0.5 * (edges[:-1] + edges[1:])
bin_id = np.clip(np.digitize(dur, edges[1:-1]), 0, len(edges) - 2)

g = excess_df.assign(_bin=bin_id).groupby("_bin")
binned = g[["observed_degree", "null_mean", "analytic_expected_degree"]].mean()
x = g["duration"].median().to_numpy()

with pp.destination("default"):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(x, binned["observed_degree"], "o-", color="k", label="observed")
    ax.plot(x, binned["null_mean"], "s--", color="dimgrey", label="windowed null")
    ax.plot(
        x,
        binned["analytic_expected_degree"],
        "^:",
        color="grey",
        label="analytic E[deg|dur]",
    )
    ax.set_xscale("log")
    ax.set_xticks([0.05, 0.1, 0.2, 0.3, 0.4])
    ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel("OFF duration (s)")
    ax.set_ylabel("Mean overlap degree")
    ax.set_title(
        "Overlap degree vs duration: observed vs chance collisions\n"
        f"scope={EXCESS_NULL_SCOPE} window={EXCESS_WINDOW:.0f}s",
        fontsize=11,
    )
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"cross_structure_5b_degree_vs_duration_{EXCESS_NULL_SCOPE}.svg"
    )
    plt.show()

## Plot 5 statistics: is observed > chance?

The location claim, that OFFs are more cross-structure-global than the windowed null
predicts (mean excess globality > 0), is asserted at the subject level and never per-OFF
(`p_greater` must not be pooled; that re-introduces pseudoreplication).
`test_excess_above_chance` reports two complementary tests:

- Paired subject-level, the headline and most conservative: per subject, mean
  `observed_degree` against mean `null_mean`, one paired value per subject, compared
  with a paired Wilcoxon signed-rank test and a paired t-test.
- Mixed model: per-`(subject, structure)` mean excess with a subject random intercept
  (`MixedLM`, structures nested in subjects), testing the fixed intercept > 0.

Plots 5a and 5b illustrate the shape and the size-dependence; this cell is where "above
chance" is formally established.


In [ ]:
# Statistical assertion: observed overlap degree > windowed-null expectation.
# Subject-level only (never pool per-OFF p_greater -> pseudoreplication).
excess_stats = cross_structure_offs.test_excess_above_chance(excess_df)

ps = excess_stats["per_subject"]
print(
    f"n subjects = {excess_stats['n_subjects']} | scope={EXCESS_NULL_SCOPE} "
    f"window={EXCESS_WINDOW:.0f}s\n"
)
print(
    ps[["subject", "mean_observed", "mean_null", "mean_excess"]]
    .round(4)
    .to_string(index=False)
)

w = excess_stats["paired_wilcoxon"]
t = excess_stats["paired_t"]
m = excess_stats["mixedlm"]
print(
    f"\nPaired Wilcoxon (observed > null): W={w['statistic']:.0f}, "
    f"p={w['p_value']:.4g}"
)
print(f"Paired t({t['df']}): t={t['statistic']:.3f}, p={t['p_value']:.4g}")
if "intercept" in m:
    print(
        f"MixedLM intercept (mean excess) = {m['intercept']:.4f} "
        f"(SE {m['se']:.4f}, z={m['z']:.2f}, p={m['p_value']:.4g}; "
        f"{m['n_units']} units / {m['n_groups']} subjects, "
        f"converged={m['converged']})"
    )
else:
    print("MixedLM:", m.get("error"))